# NLP Sprint 04 — From Counts to Embeddings

**Time limit:** 60 minutes

You will solve 60 questions:
- 20 × add-one bigram language model
- 20 × PPMI + cosine similarity
- 20 × dense-vector analogies

### Grading conventions
- lowercase all text
- tokenizer: regex `[a-z]+`
- never cross sentence boundaries
- no external data or pretrained embeddings
- final `label` must be one of `0,1,2,3`


In [ ]:
from pathlib import Path
from collections import Counter
import math, re
import numpy as np
import pandas as pd

candidates = list(Path("/kaggle/input").rglob("test.csv")) if Path("/kaggle/input").exists() else []
DATA_DIR = candidates[0].parent if candidates else Path(".")

test = pd.read_csv(DATA_DIR / "test.csv")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")
test.head()


In [ ]:
TOKEN_RE = re.compile(r"[a-z]+")

def tokenize(text):
    return TOKEN_RE.findall(str(text).lower())


## Task 1 — Add-one Bigram LM

Train on `lm_corpus.txt`.

For a sentence, use:
`<s> w1 ... wn </s>`

Prediction vocabulary:
**all lexical word types + `</s>`**.

`<s>` is a context token only and is not included in \(V\).

\[
P(w_i|w_{i-1})=\frac{C(w_{i-1},w_i)+1}{C(w_{i-1})+V}
\]

Score each candidate by average negative log-likelihood, including prediction of `</s>`.
Choose the **lowest** score.


In [ ]:
lm_lines = (DATA_DIR / "lm_corpus.txt").read_text(encoding="utf-8").splitlines()

def build_bigram_model(lines):
    # TODO 1: return bigram_counts, context_counts, V
    raise NotImplementedError

def average_sentence_nll(sentence, bigram_counts, context_counts, V):
    # TODO 2
    raise NotImplementedError

bigram_counts, context_counts, V = build_bigram_model(lm_lines)


In [ ]:
pred = {}

for _, row in test[test.task == "bigram"].iterrows():
    options = [row[f"option_{i}"] for i in range(4)]
    scores = [
        average_sentence_nll(s, bigram_counts, context_counts, V)
        for s in options
    ]
    pred[row["id"]] = int(np.argmin(scores))


## Task 2 — Term-context → PPMI → cosine

Train on `semantic_corpus.txt`.

Build a sorted vocabulary and a symmetric \(\pm 2\) context-window matrix.
The target word itself is not counted as context.

\[
PPMI(w,c)=\max\left(
\log_2\frac{P(w,c)}{P(w)P(c)},0
\right)
\]

Represent each word by its PPMI row.
Choose the candidate with the **largest cosine similarity**.

If either vector has norm 0, define cosine as 0.


In [ ]:
semantic_lines = (DATA_DIR / "semantic_corpus.txt").read_text(encoding="utf-8").splitlines()

def build_term_context(lines, window=2):
    # TODO 3: return vocab, word_to_id, F
    raise NotImplementedError

def compute_ppmi(F):
    # TODO 4
    raise NotImplementedError

def cosine(u, v):
    # TODO 5
    raise NotImplementedError

vocab, word_to_id, F = build_term_context(semantic_lines, window=2)
M = compute_ppmi(F)


In [ ]:
for _, row in test[test.task == "ppmi"].iterrows():
    target = row["arg1"]
    options = [row[f"option_{i}"] for i in range(4)]
    u = M[word_to_id[target]]
    scores = [cosine(u, M[word_to_id[w]]) for w in options]
    pred[row["id"]] = int(np.argmax(scores))


## Task 3 — Dense embedding analogies

Use `embeddings.csv`.

For:

`a : b :: c : ?`

compute

\[
q=v_b-v_a+v_c
\]

and choose the candidate with largest cosine similarity to \(q\).


In [ ]:
emb_df = pd.read_csv(DATA_DIR / "embeddings.csv")
embedding = {
    row["word"]: row.drop(labels=["word"]).to_numpy(dtype=float)
    for _, row in emb_df.iterrows()
}

def solve_analogy(a, b, c, options, embedding):
    # TODO 6
    raise NotImplementedError

for _, row in test[test.task == "analogy"].iterrows():
    options = [row[f"option_{i}"] for i in range(4)]
    pred[row["id"]] = solve_analogy(
        row["arg1"], row["arg2"], row["arg3"], options, embedding
    )


## Create `submission.csv`


In [ ]:
submission = sample_submission.copy()
submission["label"] = submission["id"].map(pred)

assert submission["label"].notna().all(), "Missing predictions."
submission["label"] = submission["label"].astype(int)
assert submission["label"].isin([0,1,2,3]).all()

submission.to_csv("submission.csv", index=False)
submission.head(), submission.shape
